# 01 – Exploratory Data Analysis

**Project**: DengAI – Predicting Disease Spread  
**Author**: Jarret  
**Competition**: https://www.drivendata.org/competitions/44/

---

### Objective
Understand the dengue fever dataset:
- Data structure, quality, and missingness
- Target variable distribution (total cases per week)
- Temporal patterns and seasonality
- Feature correlations with dengue cases
- City-level differences (San Juan vs Iquitos)

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

ROOT = Path('../')
RAW  = ROOT / 'data/raw'

features = pd.read_csv(RAW / 'training_set_features.csv', parse_dates=['week_start_date'])
labels   = pd.read_csv(RAW / 'training_set_labels.csv')
test_raw = pd.read_csv(RAW / 'test_set_features.csv',    parse_dates=['week_start_date'])

df = features.merge(labels, on=['city','year','weekofyear'])
df = df.sort_values(['city','week_start_date']).reset_index(drop=True)
print(f"Train: {df.shape}  |  Test: {test_raw.shape}")
print(df.head(3))

Train: (1456, 25)  |  Test: (416, 24)
  city  year  weekofyear week_start_date   ndvi_ne   ndvi_nw   ndvi_se  \
0   iq  2000          26      2000-07-01  0.192886  0.132257  0.340886   
1   iq  2000          27      2000-07-08  0.216833  0.276100  0.289457   
2   iq  2000          28      2000-07-15  0.176757  0.173129  0.204114   

    ndvi_sw  precipitation_amt_mm  reanalysis_air_temp_k  ...  \
0  0.247200                 25.41             296.740000  ...   
1  0.241657                 60.61             296.634286  ...   
2  0.128014                 55.52             296.415714  ...   

   reanalysis_relative_humidity_percent  reanalysis_sat_precip_amt_mm  \
0                             92.418571                         25.41   
1                             93.581429                         60.61   
2                             95.848571                         55.52   

   reanalysis_specific_humidity_g_per_kg  reanalysis_tdtr_k  \
0                              16.651429        

In [2]:
# --- Missing values ---
miss = features.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print("Missing value counts:")
print(miss.to_string())

Missing value counts:
ndvi_ne                                  194
ndvi_nw                                   52
station_diur_temp_rng_c                   43
station_avg_temp_c                        43
station_precip_mm                         22
ndvi_sw                                   22
ndvi_se                                   22
station_max_temp_c                        20
station_min_temp_c                        14
precipitation_amt_mm                      13
reanalysis_sat_precip_amt_mm              13
reanalysis_air_temp_k                     10
reanalysis_avg_temp_k                     10
reanalysis_dew_point_temp_k               10
reanalysis_max_air_temp_k                 10
reanalysis_min_air_temp_k                 10
reanalysis_relative_humidity_percent      10
reanalysis_specific_humidity_g_per_kg     10
reanalysis_tdtr_k                         10
reanalysis_precip_amt_kg_per_m2           10


In [3]:
# --- Target stats by city ---
print(df.groupby('city')['total_cases'].agg(['count','mean','median','std','max']).round(1))

      count  mean  median   std  max
city                                
iq      520   7.6     5.0  10.8  116
sj      936  34.2    19.0  51.4  461


In [4]:
# --- EDA Figure 1: Distribution + Time Series ---
sj = df[df.city == 'sj']
iq = df[df.city == 'iq']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('DengAI – EDA Overview', fontsize=14, fontweight='bold')

for ax, (city, cdf, color) in zip(axes[0], [('San Juan', sj, '#1f77b4'), ('Iquitos', iq, '#ff7f0e')]):
    ax.hist(cdf['total_cases'], bins=40, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(cdf['total_cases'].median(), color='red', linestyle='--',
               label=f"Median={cdf['total_cases'].median():.0f}")
    ax.set_title(f'{city} — Case Distribution')
    ax.set_xlabel('Weekly Cases'); ax.set_ylabel('Frequency'); ax.legend()

for ax, (city, cdf, color) in zip(axes[1], [('San Juan', sj, '#1f77b4'), ('Iquitos', iq, '#ff7f0e')]):
    ax.plot(cdf['week_start_date'], cdf['total_cases'], color=color, alpha=0.7, linewidth=0.8)
    ax.fill_between(cdf['week_start_date'], cdf['total_cases'], alpha=0.2, color=color)
    ax.set_title(f'{city} — Weekly Cases Over Time')
    ax.set_xlabel('Date'); ax.set_ylabel('Cases')

plt.tight_layout()
plt.savefig('../reports/figures/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: eda_overview.png")

Saved: eda_overview.png


In [5]:
# --- EDA Figure 2: Feature correlations ---
num_cols = [c for c in df.select_dtypes(include='number').columns
            if c not in ['year','weekofyear','total_cases']]

corr_sj = sj[num_cols + ['total_cases']].corr()['total_cases'].drop('total_cases')
corr_iq = iq[num_cols + ['total_cases']].corr()['total_cases'].drop('total_cases')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, corr, city in zip(axes, [corr_sj, corr_iq], ['San Juan', 'Iquitos']):
    top = corr.abs().nlargest(12)
    vals = corr[top.index]
    colors = ['#d62728' if v > 0 else '#1f77b4' for v in vals]
    ax.barh(range(len(vals)), vals.values, color=colors, alpha=0.75)
    ax.set_yticks(range(len(vals))); ax.set_yticklabels(vals.index, fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{city} — Feature Correlations with Total Cases')
    ax.set_xlabel('Pearson r')

plt.tight_layout()
plt.savefig('../reports/figures/eda_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## Key EDA Findings

| Finding | San Juan | Iquitos |
|---------|----------|---------|
| Mean weekly cases | 34.2 | 7.6 |
| Median weekly cases | 19.0 | 5.0 |
| Peak cases | 461 | 116 |
| Years covered | 1990–2008 | 2000–2010 |

**Target distribution**: Highly right-skewed in both cities — most weeks have low case counts, but occasional outbreaks create heavy tails. San Juan outbreaks are far larger in absolute terms.

**Seasonality**: Both cities show clear seasonal cycles. San Juan peaks in late summer/fall; Iquitos follows a different wet-season pattern.

**Top correlated features** (with `total_cases`):
- **San Juan**: humidity (reanalysis + station), minimum temperature, dew point
- **Iquitos**: specific humidity, dew point, vegetation (NDVI)

**Missing data**: 20 columns have gaps; `ndvi_ne` worst at 194 rows. Forward-fill is appropriate given the temporal structure.

**Modelling implication**: Cities are different enough that city-specific models will outperform a single joint model.